# Step 3c — Provable linear culture erasure (LEACE)

Step 3b showed the GRL never removes culture (fresh probes stay at ~99%). This notebook uses **LEACE** (Belrose et al., NeurIPS 2023), a closed-form eraser that removes **all linearly decodable culture information** from the input features, fitted on each training fold only.

It answers the question: *if culture really is removed, does fairness improve?*

Runs per backbone (same 5×5 grouped CV as before):
| run | culture erased from input | culture FiLM | soft labels |
|---|---|---|---|
| `plain` | no | no | no |
| `leace` | yes | no | no |
| `cond_soft` | no | yes | yes |
| `leace_cond_soft` | yes | yes | yes |

`leace_cond_soft` is the cleanest version of the "separate content from cultural perspective" idea: culture is erased from the meme content, then added back only through the explicit culture embedding.

**Kaggle:** GPU T4, Internet On, **Save Version → Save & Run All** (~1 h).

In [1]:
import glob, json, math, os, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from types import SimpleNamespace
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.neural_network import MLPClassifier
import warnings; warnings.filterwarnings("ignore", category=UserWarning)

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True) + glob.glob(f"/kaggle/working/**/{name}", recursive=True)
    return hits[0] if hits else None

FEATURES = {"openclip_b32": find("feats_openclip_b32.npz"), "siglip2": find("feats_siglip2.npz")}
print("Attached inputs:", os.listdir("/kaggle/input") if os.path.exists("/kaggle/input") else "none")
print("Found features :", FEATURES)
if not all(FEATURES.values()):
    print("\nFeature files not found as input -> building them here (needs GPU + Internet On, ~5 min).")
RESULTS = "/kaggle/working/results_leace"
os.makedirs(RESULTS, exist_ok=True)

Attached inputs: []
Found features : {'openclip_b32': None, 'siglip2': None}

Feature files not found as input -> building them here (needs GPU + Internet On, ~5 min).


In [2]:
# ---- Build features in this notebook if they were not attached as input (same code as Step 2) ----
if not all(FEATURES.values()):
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=4.49", "open_clip_torch"], check=True)
    from PIL import Image

    REPO = "/kaggle/working/Multi3Hate"
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/MinhDucBui/Multi3Hate.git", REPO], check=True)
    DATA = f"{REPO}/data"
    LANG2CULT = {"en": "US", "de": "DE", "es": "MX", "hi": "IN", "zh": "CN"}
    final = pd.read_csv(f"{DATA}/final_annotations.csv")
    raw = pd.read_csv(f"{DATA}/raw_annotations.csv").dropna(subset=["hatespeech"])
    votes = raw.groupby(["dataset_language", "Meme ID"])["hatespeech"].agg(n_hate="sum", n_total="count")
    img_index = {(p.split("/")[-3], int(os.path.splitext(os.path.basename(p))[0])): p
                 for p in glob.glob(f"{DATA}/memes/*/*/*")}
    rows = []
    for lang, cult in LANG2CULT.items():
        for _, r in pd.read_csv(f"{DATA}/captions/{lang}.csv").iterrows():
            mid = int(r["Meme ID"])
            v = votes.loc[(lang, mid)]
            rows.append({"meme_id": mid, "culture": cult, "language": lang,
                         "image_path": img_index[(lang, mid)],
                         "text": str(r["Translation"]).replace("<sep>", " / "),
                         "n_hate": int(v["n_hate"]), "n_total": int(v["n_total"]),
                         "label": int(final.loc[final["Meme ID"] == mid, cult].iloc[0])})
    df = pd.DataFrame(rows).sort_values(["meme_id", "culture"]).reset_index(drop=True)
    df.to_csv("/kaggle/working/multi3hate.csv", index=False)
    print("table:", df.shape)

    DEVICE_FE = "cuda" if torch.cuda.is_available() else "cpu"

    def as_tensor(out):
        if isinstance(out, torch.Tensor):
            return out
        for key in ("pooler_output", "image_embeds", "text_embeds"):
            v = getattr(out, key, None)
            if isinstance(v, torch.Tensor):
                return v
        raise TypeError(type(out))

    def load_backend(name):
        if name == "siglip2":
            from transformers import AutoModel, AutoProcessor
            mid = "google/siglip2-base-patch16-224"
            model = AutoModel.from_pretrained(mid).to(DEVICE_FE).eval()
            proc = AutoProcessor.from_pretrained(mid)
            enc_img = lambda ims: as_tensor(model.get_image_features(**proc(images=ims, return_tensors="pt").to(DEVICE_FE)))
            enc_txt = lambda ts: as_tensor(model.get_text_features(**proc(
                text=[t.lower() for t in ts], padding="max_length", max_length=64,
                truncation=True, return_tensors="pt").to(DEVICE_FE)))
            return enc_img, enc_txt
        import open_clip
        model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
        model = model.to(DEVICE_FE).eval()
        tok = open_clip.get_tokenizer("ViT-B-32")
        enc_img = lambda ims: model.encode_image(torch.stack([preprocess(i) for i in ims]).to(DEVICE_FE))
        enc_txt = lambda ts: model.encode_text(tok(ts).to(DEVICE_FE))
        return enc_img, enc_txt

    @torch.no_grad()
    def extract(backend, out_path, batch=32):
        enc_img, enc_txt = load_backend(backend)
        img, txt = [], []
        for i in range(0, len(df), batch):
            chunk = df.iloc[i:i + batch]
            ims = [Image.open(p).convert("RGB") for p in chunk.image_path]
            img.append(F.normalize(enc_img(ims).float(), dim=-1).cpu().numpy())
            txt.append(F.normalize(enc_txt(chunk.text.tolist()).float(), dim=-1).cpu().numpy())
        img, txt = np.concatenate(img), np.concatenate(txt)
        np.savez(out_path, X=np.concatenate([img, txt], 1).astype(np.float32), img_dim=img.shape[1],
                 culture=df.culture.values.astype(str), meme_id=df.meme_id.values.astype(str),
                 y_soft=(df.n_hate / df.n_total).values.astype(np.float32),
                 y_hard=df.label.values.astype(np.int64), backend=backend)
        print(f"saved {out_path}: {img.shape[0]} x {img.shape[1] + txt.shape[1]}")
        torch.cuda.empty_cache()

    for bk, backend in [("openclip_b32", "openclip"), ("siglip2", "siglip2")]:
        if not FEATURES[bk]:
            path = f"/kaggle/working/feats_{bk}.npz"
            extract(backend, path)
            FEATURES[bk] = path
    import shutil; shutil.rmtree(REPO, ignore_errors=True)

print("Using features:", FEATURES)
assert all(FEATURES.values())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.2 MB/s eta 0:00:00


Cloning into '/kaggle/working/Multi3Hate'...


table: (1500, 8)


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

saved /kaggle/working/feats_openclip_b32.npz: 1500 x 1024


...

preprocessor_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/34.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

saved /kaggle/working/feats_siglip2.npz: 1500 x 1536
Using features: {'openclip_b32': '/kaggle/working/feats_openclip_b32.npz', 'siglip2': '/kaggle/working/feats_siglip2.npz'}


In [3]:
# ---- Shared evaluation metrics ----
import numpy as np
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             roc_auc_score, confusion_matrix)

CULTURES = ["US", "DE", "MX", "IN", "CN"]


def _f(x):
    """Plain float for JSON; NaN -> None."""
    if x is None:
        return None
    x = float(x)
    return None if np.isnan(x) else x


def safe_auc(y, p):
    return roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan


def safe_bal_acc(y, yhat):
    return balanced_accuracy_score(y, yhat) if len(np.unique(y)) == 2 else np.nan


def expected_calibration_error(y, p, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(p, edges[1:-1]), 0, n_bins - 1)
    ece = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.any():
            ece += m.mean() * abs(y[m].mean() - p[m].mean())
    return ece


def bernoulli_jsd(q, p, eps=1e-6):
    """Jensen-Shannon divergence (bits) between Bernoulli(q) and Bernoulli(p)."""
    q = np.clip(q, eps, 1 - eps)
    p = np.clip(p, eps, 1 - eps)
    m = 0.5 * (q + p)

    def kl(a, b):
        return a * np.log2(a / b) + (1 - a) * np.log2((1 - a) / (1 - b))

    return 0.5 * kl(q, m) + 0.5 * kl(p, m)


def _spread(vals):
    v = np.array([x for x in vals if x is not None and not np.isnan(x)], dtype=float)
    return (v.max() - v.min()) if len(v) else np.nan


def evaluate(y_true, y_prob, culture, y_soft=None, thr=0.5):
    """Full metric set: overall, per-culture, fairness gaps, calibration, soft-label fit."""
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    culture = np.asarray(culture).astype(str)
    y_pred = (y_prob >= thr).astype(int)

    out = {
        "n": int(len(y_true)),
        "threshold": _f(thr),
        "pos_rate_true": _f(y_true.mean()),
        "pos_rate_pred": _f(y_pred.mean()),
        "accuracy": _f(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": _f(safe_bal_acc(y_true, y_pred)),
        "macro_f1": _f(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_hate": _f(f1_score(y_true, y_pred, zero_division=0)),
        "auroc": _f(safe_auc(y_true, y_prob)),
        "ece": _f(expected_calibration_error(y_true, y_prob)),
        "constant_all_hate_acc": _f(y_true.mean()),  # reference only
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=[0, 1]).tolist(),
    }
    out["degenerate"] = bool(out["pos_rate_pred"] > 0.95 or out["pos_rate_pred"] < 0.05)

    per = {}
    for c in [k for k in CULTURES if k in set(culture)] + sorted(set(culture) - set(CULTURES)):
        m = culture == c
        yt, yp = y_true[m], y_pred[m]
        per[c] = {
            "n": int(m.sum()),
            "hate_prevalence": _f(yt.mean()),
            "accuracy": _f((yt == yp).mean()),
            "balanced_accuracy": _f(safe_bal_acc(yt, yp)),
            "tpr": _f(yp[yt == 1].mean()) if (yt == 1).any() else None,
            "fpr": _f(yp[yt == 0].mean()) if (yt == 0).any() else None,
            "pos_rate_pred": _f(yp.mean()),
        }
    out["per_culture"] = per

    def col(key):
        return [np.nan if v[key] is None else v[key] for v in per.values()]

    accs, bals = col("accuracy"), col("balanced_accuracy")
    out["worst_culture_acc"] = _f(np.nanmin(accs))
    out["worst_culture_bal_acc"] = _f(np.nanmin(bals)) if not np.all(np.isnan(bals)) else None
    out["bias_gap_acc"] = _f(_spread(accs))
    out["bias_gap_bal_acc"] = _f(_spread(bals))
    out["tpr_gap"] = _f(_spread(col("tpr")))
    out["fpr_gap"] = _f(_spread(col("fpr")))
    gaps = [g for g in (out["tpr_gap"], out["fpr_gap"]) if g is not None]
    out["equalized_odds_gap"] = _f(max(gaps)) if gaps else None
    # What a constant "hate" predictor would score as a bias gap on this split:
    out["constant_predictor_bias_gap"] = _f(_spread(col("hate_prevalence")))

    if y_soft is not None:
        q = np.asarray(y_soft, dtype=float)
        p = np.clip(y_prob, 1e-6, 1 - 1e-6)
        out["soft_bce"] = _f(np.mean(-(q * np.log(p) + (1 - q) * np.log(1 - p))))
        out["soft_jsd"] = _f(np.mean(bernoulli_jsd(q, y_prob)))
    return out

In [4]:
# ---- LEACE: closed-form least-squares concept erasure (Belrose et al., 2023) ----
def fit_leace(X, c, n_cls, eps=1e-6):
    """Returns (mu, E) such that x' = x - (x - mu) @ E.T removes all linear information about c."""
    X = X.astype(np.float64)
    mu = X.mean(0)
    Xc = X - mu
    Z = np.eye(n_cls)[c]
    Zc = Z - Z.mean(0)
    n = len(X)
    Sxx = Xc.T @ Xc / n
    Sxz = Xc.T @ Zc / n
    evals, evecs = np.linalg.eigh(Sxx)
    keep = evals > eps * evals.max()
    V, ev = evecs[:, keep], evals[keep]
    W = V @ np.diag(ev ** -0.5) @ V.T          # whitening
    W_pinv = V @ np.diag(ev ** 0.5) @ V.T      # its pseudo-inverse
    U, s, _ = np.linalg.svd(W @ Sxz, full_matrices=False)
    U = U[:, s > 1e-8 * s.max()]
    E = W_pinv @ (U @ U.T) @ W
    return mu, E

def apply_leace(X, mu, E):
    return (X - (X - mu) @ E.T).astype(np.float32)

In [5]:
# ---- Model, training, probes, baselines ----
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lam):
        ctx.lam = lam
        return x.view_as(x)

    @staticmethod
    def backward(ctx, g):
        return -ctx.lam * g, None


class CCAL(nn.Module):
    def __init__(self, in_dim, hid, n_cult, cult_dim, p, adv, conditioned):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(in_dim, hid), nn.ReLU(), nn.Dropout(p),
                                 nn.Linear(hid, hid), nn.ReLU(), nn.Dropout(p))
        self.conditioned = conditioned
        if conditioned:
            self.cult_emb = nn.Embedding(n_cult, cult_dim)
            self.film = nn.Linear(cult_dim, 2 * hid)
            nn.init.zeros_(self.film.weight); nn.init.zeros_(self.film.bias)  # start as identity
        self.head = nn.Sequential(nn.Linear(hid, 128), nn.ReLU(), nn.Dropout(p), nn.Linear(128, 1))
        self.adv_type = adv
        if adv != "none":
            extra = 2 if adv == "cond_grl" else 0
            self.adv = nn.Sequential(nn.Linear(hid + extra, 128), nn.ReLU(), nn.Dropout(p),
                                     nn.Linear(128, n_cult))

    def forward(self, x, c, y_hard=None, lam=0.0, with_adv=False):
        z = self.enc(x)
        h = z
        if self.conditioned:
            g, b = self.film(self.cult_emb(c)).chunk(2, dim=-1)
            h = z * (1 + g) + b
        logit = self.head(h).squeeze(-1)
        adv_logit = None
        if with_adv and self.adv_type != "none":
            zr = GradReverse.apply(z, lam)
            if self.adv_type == "cond_grl":
                zr = torch.cat([zr, F.one_hot(y_hard, 2).float()], dim=-1)
            adv_logit = self.adv(zr)
        return logit, adv_logit, z


def lam_at(epoch, total, lam_max):
    p = epoch / max(total - 1, 1)
    return lam_max * (2.0 / (1.0 + math.exp(-10.0 * p)) - 1.0)


@torch.no_grad()
def predict(model, X, C, bs=512):
    model.eval()
    probs, zs = [], []
    for i in range(0, len(X), bs):
        x = torch.as_tensor(X[i:i + bs], dtype=torch.float32, device=DEV)
        c = torch.as_tensor(C[i:i + bs], dtype=torch.long, device=DEV)
        logit, _, z = model(x, c)
        probs.append(torch.sigmoid(logit).cpu().numpy()); zs.append(z.cpu().numpy())
    return np.concatenate(probs), np.concatenate(zs)


def tune_threshold(y, p):
    grid = np.linspace(0.05, 0.95, 91)
    scores = [balanced_accuracy_score(y, (p >= t).astype(int)) for t in grid]
    return float(grid[int(np.argmax(scores))])


def train_one(d, tr, va, lam_max, a, seed):
    set_seed(seed)
    X, C, YH, YS = d["X"], d["c"], d["yh"], d["ys"]
    model = CCAL(X.shape[1], a.hidden, len(CULTURES), a.cult_dim, a.dropout,
                 a.adv, not a.no_condition).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=a.wd)
    target = YS if a.soft else YH.astype(np.float32)
    n_pos = YH[tr].sum()
    pos_weight = torch.tensor((len(tr) - n_pos) / max(n_pos, 1), dtype=torch.float32, device=DEV)

    ds = TensorDataset(torch.tensor(X[tr], dtype=torch.float32), torch.tensor(C[tr]),
                       torch.tensor(YH[tr]), torch.tensor(target[tr], dtype=torch.float32))
    dl = DataLoader(ds, batch_size=a.batch, shuffle=True,
                    generator=torch.Generator().manual_seed(seed))

    best, best_state, best_thr, bad = -np.inf, None, 0.5, 0
    for ep in range(a.epochs):
        lam = lam_at(ep, a.epochs, lam_max) if a.adv != "none" else 0.0
        model.train()
        for xb, cb, yhb, tb in dl:
            xb, cb, yhb, tb = xb.to(DEV), cb.to(DEV), yhb.to(DEV), tb.to(DEV)
            logit, adv_logit, _ = model(xb, cb, yhb, lam, with_adv=a.adv != "none")
            loss = F.binary_cross_entropy_with_logits(logit, tb, pos_weight=pos_weight)
            if adv_logit is not None:
                # DANN convention: adversary trained at full strength,
                # encoder receives -lambda * gradient through the GRL.
                loss = loss + F.cross_entropy(adv_logit, cb)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        # Model selection starts only after min_epochs, so the chosen checkpoint
        # has the adversary substantially ramped up (not an early, un-debiased one).
        if ep < a.min_epochs and ep < a.epochs - 1:
            continue
        p_va, _ = predict(model, X[va], C[va])
        thr = tune_threshold(YH[va], p_va)
        score = evaluate(YH[va], p_va, d["cname"][va], YS[va], thr)[a.select]
        score = -np.inf if score is None else score
        if best_state is None or score > best + 1e-4:
            best, best_thr, bad = score, thr, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= a.patience:
                break
    model.load_state_dict(best_state)
    return model, best, best_thr


def leakage_probes(model, d, tr, te, seed):
    _, ztr = predict(model, d["X"][tr], d["c"][tr])
    _, zte = predict(model, d["X"][te], d["c"][te])
    lin = LogisticRegression(max_iter=3000).fit(ztr, d["c"][tr]).score(zte, d["c"][te])
    mlp = MLPClassifier((128,), max_iter=600, random_state=seed).fit(ztr, d["c"][tr]).score(zte, d["c"][te])
    raw = LogisticRegression(max_iter=3000).fit(d["X"][tr], d["c"][tr]).score(d["X"][te], d["c"][te])
    return {"probe_z_linear_acc": float(lin), "probe_z_mlp_acc": float(mlp),
            "probe_input_linear_acc": float(raw), "probe_chance": 1 / len(CULTURES)}


def baselines(d, tr, te):
    yh, c = d["yh"], d["c"]
    const = np.full(len(te), float(yh[tr].mean() >= 0.5))
    per = np.array([yh[tr][c[tr] == k].mean() for k in range(len(CULTURES))])
    return {
        "majority": evaluate(yh[te], const, d["cname"][te], d["ys"][te]),
        "per_culture_majority": evaluate(yh[te], per[c[te]], d["cname"][te], d["ys"][te]),
    }


def load(path, modality):
    z = np.load(path, allow_pickle=True)
    X = z["X"].astype(np.float32)
    k = int(z["img_dim"])
    X = {"both": X, "image": X[:, :k], "text": X[:, k:]}[modality]
    cname = z["culture"].astype(str)
    return dict(X=X, cname=cname, c=np.array([CULTURES.index(s) for s in cname]),
                meme=z["meme_id"].astype(str), ys=z["y_soft"].astype(np.float32),
                yh=z["y_hard"].astype(np.int64))


SUMMARY_KEYS = ["accuracy", "balanced_accuracy", "macro_f1", "auroc", "ece",
                "worst_culture_acc", "worst_culture_bal_acc", "bias_gap_acc",
                "bias_gap_bal_acc", "equalized_odds_gap", "pos_rate_pred",
                "soft_bce", "soft_jsd", "probe_z_linear_acc", "probe_z_mlp_acc",
                "probe_input_linear_acc"]


def summarize(rows):
    s = {}
    for k in SUMMARY_KEYS:
        v = np.array([r[k] for r in rows if r.get(k) is not None], dtype=float)
        if len(v):
            s[k] = {"mean": float(v.mean()), "std": float(v.std(ddof=1)) if len(v) > 1 else 0.0}
    s["degenerate_folds"] = int(sum(r["degenerate"] for r in rows))
    s["n_folds"] = len(rows)
    return s

In [6]:
DEFAULTS = dict(adv="cond_grl", no_condition=False, soft=False, modality="both",
                lam_grid="0.05,0.1,0.2,0.5", select="balanced_accuracy",
                folds=5, repeats=5, epochs=60, min_epochs=15, patience=10, batch=32,
                hidden=256, cult_dim=32, dropout=0.3, lr=1e-4, wd=1e-4, seed=42, erase=False)

def run_experiment(name, features, **overrides):
    """Runs repeated grouped CV for one configuration and saves results/<name>/."""
    out = os.path.join(RESULTS, name)
    if os.path.exists(os.path.join(out, "summary.json")):
        print(f"[skip] {name} already done")
        return json.load(open(os.path.join(out, "summary.json")))
    a = SimpleNamespace(**{**DEFAULTS, **overrides})
    os.makedirs(out, exist_ok=True)
    d = load(features, a.modality)
    lam_grid = [0.0] if a.adv == "none" else [float(x) for x in a.lam_grid.split(",")]
    fold_rows, base_rows, preds = [], {"majority": [], "per_culture_majority": []}, []
    t0 = time.time()
    print(f"\n===== {name} =====")
    for rep in range(a.repeats):
        sgkf = StratifiedGroupKFold(n_splits=a.folds, shuffle=True, random_state=1000 + rep)
        for fold, (tr_all, te) in enumerate(sgkf.split(d["X"], d["yh"], groups=d["meme"])):
            seed = a.seed + 100 * rep + fold
            gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=seed)
            itr, iva = next(gss.split(tr_all, d["yh"][tr_all], groups=d["meme"][tr_all]))
            tr, va = tr_all[itr], tr_all[iva]
            d_orig = d
            if a.erase:  # fit the eraser on the outer training fold only, apply everywhere
                mu, E = fit_leace(d["X"][tr_all], d["c"][tr_all], len(CULTURES))
                d = dict(d, X=apply_leace(d["X"], mu, E))
            best = None
            for lam in lam_grid:
                model, score, thr = train_one(d, tr, va, lam, a, seed)
                if best is None or score > best[1]:
                    best = (model, score, thr, lam)
            model, _, thr, lam = best
            p_te, _ = predict(model, d["X"][te], d["c"][te])
            m = evaluate(d["yh"][te], p_te, d["cname"][te], d["ys"][te], thr)
            m.update(leakage_probes(model, d, tr, te, seed))
            m.update({"repeat": rep, "fold": fold, "lambda_max": lam})
            fold_rows.append(m)
            for k, v in baselines(d, tr_all, te).items():
                base_rows[k].append(v)
            preds.append(pd.DataFrame({"repeat": rep, "fold": fold, "meme_id": d["meme"][te],
                                       "culture": d["cname"][te], "y_true": d["yh"][te],
                                       "y_soft": d["ys"][te], "y_prob": p_te,
                                       "y_pred": (p_te >= thr).astype(int)}))
            d = d_orig
            f3 = lambda v: "nan" if v is None else f"{v:.3f}"
            print(f"rep {rep} fold {fold} | lam={lam:.2f} thr={thr:.2f} | bal_acc={f3(m['balanced_accuracy'])} "
                  f"auroc={f3(m['auroc'])} worst={f3(m['worst_culture_bal_acc'])} "
                  f"gap={f3(m['bias_gap_bal_acc'])} probe_z={f3(m['probe_z_mlp_acc'])}"
                  f"{'  <-- DEGENERATE' if m['degenerate'] else ''}")
    summary = {"name": name, "features": os.path.basename(features), "config": vars(a),
               "model": summarize(fold_rows),
               "baselines": {k: summarize(v) for k, v in base_rows.items()},
               "minutes": round((time.time() - t0) / 60, 1)}
    json.dump({"model": fold_rows, "baselines": base_rows}, open(os.path.join(out, "fold_metrics.json"), "w"), indent=1)
    pd.concat(preds).to_csv(os.path.join(out, "predictions.csv"), index=False)
    json.dump(summary, open(os.path.join(out, "summary.json"), "w"), indent=1)  # written last = "done" marker
    s = summary["model"]
    print(f"--> {name}: bal_acc {s['balanced_accuracy']['mean']:.3f}±{s['balanced_accuracy']['std']:.3f} | "
          f"worst-culture {s['worst_culture_bal_acc']['mean']:.3f} | gap {s['bias_gap_bal_acc']['mean']:.3f} | "
          f"degenerate {s['degenerate_folds']}/{s['n_folds']} | {summary['minutes']} min")
    return summary

### Quick test (optional)
Set `QUICK_TEST = True` to check everything runs in a couple of minutes before the full matrix. Set it back to `False` for the real run.

In [7]:
QUICK_TEST = False
if QUICK_TEST:
    run_experiment("_quicktest", FEATURES["siglip2"], adv="none", soft=True, erase=True,
                   folds=3, repeats=1, epochs=20, min_epochs=5, lam_grid="0.1")
    import shutil; shutil.rmtree(os.path.join(RESULTS, "_quicktest"), ignore_errors=True)

### Full experiment matrix

In [8]:
VARIANTS = [
    ("plain",           dict(adv="none", no_condition=True,  soft=False, erase=False)),
    ("leace",           dict(adv="none", no_condition=True,  soft=False, erase=True)),
    ("cond_soft",       dict(adv="none", no_condition=False, soft=True,  erase=False)),
    ("leace_cond_soft", dict(adv="none", no_condition=False, soft=True,  erase=True)),
]
if not QUICK_TEST:
    for bk, path in FEATURES.items():
        for tag, cfg in VARIANTS:
            run_experiment(f"{bk}_{tag}", path, **cfg)


===== openclip_b32_plain =====
rep 0 fold 0 | lam=0.00 thr=0.55 | bal_acc=0.596 auroc=0.629 worst=0.542 gap=0.068 probe_z=1.000
rep 0 fold 1 | lam=0.00 thr=0.92 | bal_acc=0.537 auroc=0.596 worst=0.500 gap=0.091 probe_z=0.990
rep 0 fold 2 | lam=0.00 thr=0.29 | bal_acc=0.586 auroc=0.624 worst=0.546 gap=0.059 probe_z=0.977
rep 0 fold 3 | lam=0.00 thr=0.18 | bal_acc=0.538 auroc=0.549 worst=0.496 gap=0.092 probe_z=0.983
rep 0 fold 4 | lam=0.00 thr=0.53 | bal_acc=0.588 auroc=0.609 worst=0.488 gap=0.179 probe_z=0.993
rep 1 fold 0 | lam=0.00 thr=0.70 | bal_acc=0.497 auroc=0.606 worst=0.444 gap=0.097 probe_z=0.990
rep 1 fold 1 | lam=0.00 thr=0.73 | bal_acc=0.589 auroc=0.642 worst=0.548 gap=0.091 probe_z=0.987
rep 1 fold 2 | lam=0.00 thr=0.61 | bal_acc=0.593 auroc=0.605 worst=0.493 gap=0.153 probe_z=0.983
rep 1 fold 3 | lam=0.00 thr=0.83 | bal_acc=0.528 auroc=0.622 worst=0.460 gap=0.129 probe_z=0.983
rep 1 fold 4 | lam=0.00 thr=0.28 | bal_acc=0.502 auroc=0.555 worst=0.461 gap=0.073 probe_z=0.98

### Results table

In [9]:
KEYS = ["accuracy", "balanced_accuracy", "macro_f1", "auroc", "worst_culture_bal_acc",
        "bias_gap_bal_acc", "equalized_odds_gap", "ece", "soft_jsd", "probe_input_linear_acc", "probe_z_linear_acc", "probe_z_mlp_acc"]
rows = []
for f in sorted(glob.glob(f"{RESULTS}/*/summary.json")):
    s = json.load(open(f))
    r = {"run": s["name"]}
    for k in KEYS:
        v = s["model"].get(k)
        r[k] = f"{v['mean']:.3f} ± {v['std']:.3f}" if v else ""
    r["degenerate"] = f"{s['model']['degenerate_folds']}/{s['model']['n_folds']}"
    rows.append(r)
    if s["name"].endswith("_plain"):   # add train-set majority baselines once per backbone
        for bname, bs in s["baselines"].items():
            rb = {"run": s["name"].replace("plain", bname)}
            for k in KEYS:
                v = bs.get(k); rb[k] = f"{v['mean']:.3f} ± {v['std']:.3f}" if v else ""
            rb["degenerate"] = f"{bs['degenerate_folds']}/{bs['n_folds']}"
            rows.append(rb)
table = pd.DataFrame(rows)
table.to_csv(f"{RESULTS}/results_table.csv", index=False)
pd.set_option("display.max_columns", None); pd.set_option("display.width", 250)
table

,run,accuracy,balanced_accuracy,macro_f1,auroc,worst_culture_bal_acc,bias_gap_bal_acc,equalized_odds_gap,ece,soft_jsd,probe_input_linear_acc,probe_z_linear_acc,probe_z_mlp_acc,degenerate
0,openclip_b32_cond_soft,0.577 ± 0.040,0.574 ± 0.024,0.561 ± 0.038,0.614 ± 0.036,0.503 ± 0.030,0.135 ± 0.041,0.330 ± 0.084,0.162 ± 0.041,0.159 ± 0.016,0.994 ± 0.005,0.940 ± 0.029,0.985 ± 0.009,0/25
1,openclip_b32_leace,0.544 ± 0.044,0.552 ± 0.028,0.531 ± 0.041,0.589 ± 0.039,0.484 ± 0.038,0.131 ± 0.049,0.240 ± 0.065,0.235 ± 0.057,0.193 ± 0.029,0.272 ± 0.029,0.254 ± 0.027,0.228 ± 0.019,0/25
2,openclip_b32_leace_cond_soft,0.570 ± 0.043,0.569 ± 0.026,0.555 ± 0.043,0.606 ± 0.037,0.507 ± 0.034,0.116 ± 0.040,0.287 ± 0.078,0.168 ± 0.044,0.162 ± 0.016,0.272 ± 0.029,0.264 ± 0.023,0.273 ± 0.048,0/25
3,openclip_b32_plain,0.552 ± 0.049,0.560 ± 0.037,0.539 ± 0.051,0.598 ± 0.037,0.498 ± 0.045,0.114 ± 0.046,0.288 ± 0.076,0.224 ± 0.061,0.186 ± 0.032,0.994 ± 0.005,0.947 ± 0.018,0.986 ± 0.009,0/25
4,openclip_b32_majority,0.580 ± 0.051,0.500 ± 0.000,0.366 ± 0.020,0.500 ± 0.000,0.500 ± 0.000,0.000 ± 0.000,0.000 ± 0.000,0.420 ± 0.051,0.354 ± 0.041,,,,25/25
5,openclip_b32_per_culture_majority,0.575 ± 0.047,0.502 ± 0.009,0.379 ± 0.049,0.544 ± 0.021,0.500 ± 0.000,0.000 ± 0.000,0.120 ± 0.332,0.058 ± 0.028,0.148 ± 0.010,,,,22/25
6,siglip2_cond_soft,0.636 ± 0.045,0.631 ± 0.041,0.619 ± 0.047,0.689 ± 0.049,0.568 ± 0.043,0.123 ± 0.044,0.242 ± 0.066,0.127 ± 0.045,0.140 ± 0.014,0.996 ± 0.003,0.889 ± 0.044,0.992 ± 0.006,0/25
7,siglip2_leace,0.631 ± 0.040,0.630 ± 0.036,0.620 ± 0.040,0.681 ± 0.048,0.572 ± 0.042,0.120 ± 0.043,0.199 ± 0.051,0.168 ± 0.060,0.157 ± 0.026,0.254 ± 0.022,0.213 ± 0.015,0.217 ± 0.020,0/25
8,siglip2_leace_cond_soft,0.630 ± 0.042,0.626 ± 0.040,0.613 ± 0.045,0.686 ± 0.050,0.572 ± 0.042,0.110 ± 0.044,0.227 ± 0.057,0.129 ± 0.042,0.140 ± 0.014,0.254 ± 0.022,0.217 ± 0.019,0.219 ± 0.023,0/25
9,siglip2_plain,0.632 ± 0.037,0.629 ± 0.035,0.617 ± 0.038,0.685 ± 0.044,0.556 ± 0.046,0.144 ± 0.048,0.256 ± 0.068,0.170 ± 0.058,0.158 ± 0.028,0.996 ± 0.003,0.905 ± 0.025,0.990 ± 0.008,0/25


In [10]:
import shutil
shutil.make_archive("/kaggle/working/ccal_leace", "zip", RESULTS)
print("Download /kaggle/working/ccal_leace.zip from the Output tab.")

Download /kaggle/working/ccal_leace.zip from the Output tab.


## What to send back
Upload **`ccal_leace.zip`** from the Output tab. The key columns are `probe_input_linear_acc` (should drop to ~0.20 for the `leace` runs, confirming erasure), `probe_z_mlp_acc` (non-linear leakage that remains), and the accuracy/fairness columns.